# Multi-Injection Pipeline with Rubin PSF and Diagnostics (RSP)

RSP-ready copy. Injects clusters in batches with the Rubin CoaddPsf,
produces postage stamps, diagnostic plots, and completeness plots.

In [ ]:
# ---- RSP setup ----
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm, AsinhNorm
from matplotlib.patches import Circle
from datetime import datetime

USERNAME = os.environ.get('USER', 'your_username')
PIPELINE_PATH = f'/home/{USERNAME}/WORK/INJECT/star-cluster-injection-pipeline'

if os.path.exists(PIPELINE_PATH):
    print(f'✓ Pipeline found at: {PIPELINE_PATH}')
    sys.path.insert(0, PIPELINE_PATH)
else:
    print(f'✗ Pipeline NOT found. Run: cd ~ && git clone https://github.com/whosneha/INJECT.git')

In [ ]:
# ---- Imports ----
from lsst.daf.butler import Butler
# NOTE: inject_clusters_in_batches does not exist in src/inject.py.
# We use inject_from_catalog and batch manually below.
from src.inject import inject_from_catalog, create_injection_catalog
from src.retrieval import ClusterRetrieval
from src.visualization import plot_postage_stamps, plot_injection_summary

output_dir = 'outputs/multi_injection_pipeline_with_diagnostics_rsp'
os.makedirs(output_dir, exist_ok=True)

butler  = Butler('dp02', collections='2.2i/runs/DP0.2')
data_id = {'tract': 3828, 'patch': 24, 'band': 'i'}
coadd   = butler.get('deepCoadd', dataId=data_id)
image   = coadd.image.array.copy()
psf_obj = coadd.getPsf()
bbox    = coadd.getBBox()
BBOX_X_MIN, BBOX_Y_MIN = bbox.getMinX(), bbox.getMinY()

print(f'Loaded deepCoadd: tract={data_id["tract"]}, patch={data_id["patch"]}, band={data_id["band"]}')
print(f'Image shape: {image.shape}')

In [ ]:
# ---- Generate injection catalog ----
n_clusters = 10000
batch_size = 1000

catalog = create_injection_catalog(
    n_clusters=n_clusters,
    image_shape=image.shape,
    mag_range=(20, 26),
    r_half_range=(2, 10),
    profile_type='king',
    edge_buffer=50,
    exposure=coadd,
    seed=42,
)
print(f'Generated catalog with {len(catalog)} clusters.')

In [ ]:
# ---- Inject in batches using inject_from_catalog ----
injected_image = image.copy()
injection_info = []

n_batches = int(np.ceil(len(catalog) / batch_size))
for b in range(n_batches):
    i0 = b * batch_size
    i1 = min(i0 + batch_size, len(catalog))
    batch = catalog[i0:i1]
    print(f'Batch {b+1}/{n_batches}: injecting {len(batch)} clusters ...')
    injected_image, info_batch = inject_from_catalog(
        injected_image, batch,
        exposure=coadd,
        add_noise=True,
        pixel_scale=0.2,
        save_stamps=True,
        psf_mode='spatially_varying',
    )
    injection_info.extend(info_batch)

np.save(os.path.join(output_dir, 'injected_image.npy'), injected_image)
print(f'✓ Injected {len(injection_info)} clusters; saved to {output_dir}/injected_image.npy')

In [ ]:
# ---- Postage stamps ----
plot_postage_stamps(injection_info, injected_image, image, n_show=6)

In [ ]:
# ---- Diagnostic plots: before/after/difference + parameter distributions ----
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
vmin, vmax = np.percentile(image, [1, 99])
axes[0].imshow(image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[0].set_title('Original')
axes[1].imshow(injected_image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[1].set_title('Injected')
diff = injected_image.astype(np.float64) - image.astype(np.float64)
dv = np.percentile(np.abs(diff), 99)
axes[2].imshow(diff, cmap='RdBu_r', origin='lower', vmin=-dv, vmax=dv)
axes[2].set_title('Difference')
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'before_after.png'), dpi=150)
plt.show()

mags = [i['magnitude'] for i in injection_info]
rhs  = [i['r_half']    for i in injection_info]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(mags, bins=30, color='steelblue', edgecolor='k')
axes[0].set_xlabel('Magnitude'); axes[0].set_ylabel('N')
axes[1].hist(rhs, bins=30, color='darkorange', edgecolor='k')
axes[1].set_xlabel('r_half (px)')
plt.suptitle('Injected parameter distributions')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'injected_params.png'), dpi=150)
plt.show()

In [ ]:
# ---- Detection placeholder ----
detections = []  # Replace with output of your detection pipeline
print('Run your detection pipeline and populate the `detections` list.')

In [ ]:
# ---- Completeness analysis ----
if detections:
    retrieval = ClusterRetrieval(injection_info, detections)
    retrieval.match_detections(match_radius=5.0)
    stats = retrieval.get_summary_statistics()

    print('=== Completeness Results ===')
    print(f'Injected  : {stats["n_injected"]}')
    print(f'Recovered : {stats["n_detected"]}')
    print(f'Completeness: {stats["overall_completeness"]:.1%}')

    plot_injection_summary(injection_info, detections, stats)

    # Completeness vs magnitude
    mag_bins = np.linspace(20, 26, 15)
    bc, comp, comp_err = retrieval.compute_completeness(parameter='magnitude', bins=mag_bins)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.step(bc, comp, where='mid', lw=2, color='royalblue', label='Completeness')
    ax.fill_between(bc, comp - comp_err, comp + comp_err, step='mid', alpha=0.3, color='royalblue')
    ax.axhline(0.5, ls='--', color='gray'); ax.axhline(0.9, ls=':', color='gray')
    ax.set_xlabel('Magnitude'); ax.set_ylabel('Completeness')
    ax.set_ylim(-0.05, 1.15); ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'completeness_vs_mag.png'), dpi=150)
    plt.show()

    # Completeness vs r_half
    rh_bins = np.linspace(2, 10, 10)
    bc, comp, comp_err = retrieval.compute_completeness(parameter='r_half', bins=rh_bins)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.step(bc, comp, where='mid', lw=2, color='darkorange', label='Completeness')
    ax.fill_between(bc, comp - comp_err, comp + comp_err, step='mid', alpha=0.3, color='darkorange')
    ax.axhline(0.5, ls='--', color='gray'); ax.axhline(0.9, ls=':', color='gray')
    ax.set_xlabel('r_half (px)'); ax.set_ylabel('Completeness')
    ax.set_ylim(-0.05, 1.15); ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'completeness_vs_rh.png'), dpi=150)
    plt.show()
else:
    print('No detections provided. Completeness analysis skipped.')